# Flight Separator — Model Training

Trains a classifier to label each telemetry row as `taxing`, `landing-takeoff`, or `flight`.

**Features used:** `altitude`, `ground_speed`, `heading`

**Config:** Place your `config.yml` next to this notebook (or at the project root) with the shape:
```yaml
database:
  username: your-db-user
  password: your-db-password
  ip_address: your-db-host
  port: '50000'
  db_name: YOUR_DB_NAME
```

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from the project root

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from db_connection_params_handler import ConnectionParamsHandler
from db_factory import make_db_manager
from db_handlers.ibm_db2.MQTT_db_access import TelemetryMQTTDatabaseAccess
from mappers.telemetry_mapper import FormattedTelemetryMapper
from logic.position_assigner_instance import position_assigner

## 1. Connect to the database

In [ ]:
CONFIG_PATH = 'config.yml'

connection = ConnectionParamsHandler(connection_filename=CONFIG_PATH)
db_manager = make_db_manager(connection)
telemetry_access = TelemetryMQTTDatabaseAccess(manager=db_manager)
print('Connected.')

## 2. Pull labelled telemetry

We use the existing `PositionAssigner` + `FormattedTelemetryMapper` pipeline
to get processed rows with the current `attitude` label already applied.
Edit the list below to cover the registrations and date ranges you want.

In [ ]:
QUERIES = [
    # (registration, start_date, end_date)
    ('I-PVLG', '2024-01-01', '2024-06-30'),
    # add more as needed
]

frames = []
for marca, start, end in QUERIES:
    print(f'Fetching {marca}  {start} → {end} ...')
    raw = telemetry_access.get_telemetry_by_marca_datetime_interval(
        marca=marca, start_datetime=start, end_datetime=end
    )
    if raw.dropna(how='all').empty:
        print('  no data')
        continue
    mapper = FormattedTelemetryMapper(df_chunk_generator=raw, position_assigner=position_assigner)
    df = mapper.new_generate_telemetry_samples()
    frames.append(df)
    print(f'  {len(df):,} rows')

data = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(data):,}')
data.head()

## 3. Clip features & prepare labels

The existing model clips these values — we apply the same bounds so the
training distribution matches what the model sees at inference time.

In [ ]:
FEATURES = ['altitude', 'ground_speed', 'heading']

# Same clipping as FlightSeparator.__extract_labeled_samples
data['altitude']     = data['altitude'].clip(-1, 30_000)
data['ground_speed'] = data['ground_speed'].clip(-1, 500)

# The attitude column from PositionAssigner is the fine-grained label.
# Map it to the coarse flight-separator classes.
PHASE_MAP = {
    'taxing':           'taxing',
    'landing-takeoff':  'landing-takeoff',
    'ascending_straight': 'flight',
    'ascending_left':   'flight',
    'ascending_right':  'flight',
    'stable_straight':  'flight',
    'stable_left':      'flight',
    'stable_right':     'flight',
    'descending_straight': 'flight',
    'descending_left':  'flight',
    'descending_right': 'flight',
}
data['phase'] = data['attitude'].map(PHASE_MAP)
data = data.dropna(subset=FEATURES + ['phase'])

print(data['phase'].value_counts())
data[FEATURES + ['phase']].head()

## 4. Train / test split

In [ ]:
X = data[FEATURES].values
y = data['phase'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

## 5. Train model

Random Forest is the same family as the existing `.pkl`. Swap in any
sklearn-compatible classifier — the save/load mechanism is identical.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluate

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted', n_jobs=-1)
print(f'5-fold CV F1 (weighted): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap='Blues')
plt.title('Flight Separator — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh()
plt.title('Feature importances')
plt.tight_layout()
plt.show()

## 7. Save artefacts

Drop the output files into `converters/flight_separator_data/` to replace the
existing model. The filename convention mirrors the existing `accuracy.pkl` variant.

In [ ]:
OUT_DIR = Path('../converters/flight_separator_data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'retrained'  # change to 'accuracy', 'memory', or 'computation_time' to replace existing

# Model
joblib.dump(model, OUT_DIR / f'{MODEL_NAME}.pkl')

# Class names  — list ordered by model.classes_ index
class_names = list(model.classes_)
with open(OUT_DIR / f'{MODEL_NAME}_class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

# Feature list
with open(OUT_DIR / f'{MODEL_NAME}_selected_features.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

print('Saved:')
for p in OUT_DIR.glob(f'{MODEL_NAME}*'):
    print(' ', p)